In [2]:
import pandas as pd
import numpy as np
import google.generativeai as genai

c:\Users\shiva\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\shiva\AppData\Local\Temp\ipykernel_130224\2428451170.py:3: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [3]:
jobs = pd.read_csv("jobs.csv")

In [5]:
jobs["job_profile"] = (
    jobs["job_title"].astype(str) + " " +
    jobs["job_description"].astype(str) + " " +
    jobs["skills_required"].astype(str) + " " +
    jobs["industry"].astype(str) + " " +
    jobs["education_required"].astype(str)
)

In [6]:
jobs['job_profile']

0       News Reporter 1 We are hiring a News Reporter ...
1       UI/UX Designer 2 We are hiring a UI/UX Designe...
2       Psychologist 3 We are hiring a Psychologist in...
3       Professor 4 We are hiring a Professor in the F...
4       Software Developer 5 We are hiring a Software ...
                              ...                        
5995    Insurance Advisor 5996 We are hiring a Insuran...
5996    DevOps Engineer 5997 We are hiring a DevOps En...
5997    Production Manager 5998 We are hiring a Produc...
5998    Chef 5999 We are hiring a Chef in the Transpor...
5999    IoT Engineer 6000 We are hiring a IoT Engineer...
Name: job_profile, Length: 6000, dtype: object

In [14]:
from sentence_transformers import SentenceTransformer, util
model = SentenceTransformer("all-mpnet-base-v2")

In [8]:
job_embeddings = model.encode(jobs["job_profile"].tolist(), convert_to_tensor=True)

In [17]:

from sentence_transformers import util
for i, row in jobs.iterrows():
    course_embedding = model.encode(row["skills_required"], convert_to_tensor=True)
    similarities = util.cos_sim(course_embedding, job_embeddings)[0]
    
    top_k = 5
    top_results = similarities.topk(k=top_k)
    
    print(f"Skills: {row['skills_required']}")
    print(f"Expected: {row['job_title']}")
    for score, idx in zip(top_results.values, top_results.indices):
        print(f"  {score:.4f} → {jobs.iloc[int(idx)]['job_title']}")
    print("-" * 50)

Skills: C++, Java, Sales Strategy
Expected: News Reporter 1
  0.4854 → Sales Executive 1917
  0.4579 → Sales Executive 5399
  0.4505 → Sales Executive 5106
  0.4496 → Sales Executive 2776
  0.4414 → Marketing Manager 479
--------------------------------------------------
Skills: Cloud Computing, Marketing, Customer Service
Expected: UI/UX Designer 2
  0.5420 → Cloud Engineer 4515
  0.5261 → Cloud Engineer 2656
  0.5240 → Cloud Engineer 3093
  0.5167 → Cloud Engineer 2584
  0.5147 → Cloud Engineer 4112
--------------------------------------------------
Skills: SQL, AutoCAD, Excel
Expected: Psychologist 3
  0.4790 → Data Analyst 4494
  0.4391 → Data Analyst 4350
  0.4307 → Data Analyst 2399
  0.4163 → Civil Engineer 989
  0.4089 → Data Analyst 721
--------------------------------------------------
Skills: Machine Operation, Leadership, Communication
Expected: Professor 4
  0.4788 → Factory Supervisor 1880
  0.4717 → Factory Supervisor 2050
  0.4630 → Production Manager 5296
  0.4618 → Fa

In [12]:
jobs

,job_id,job_title,job_description,skills_required,certifications,industry,experience_level,education_required,job_profile
0,1,News Reporter 1,We are hiring a News Reporter in the Media ind...,"C++, Java, Sales Strategy",Digital Marketing Certification,Media,Fresher,ITI,News Reporter 1 We are hiring a News Reporter ...
1,2,UI/UX Designer 2,We are hiring a UI/UX Designer in the Legal in...,"Cloud Computing, Marketing, Customer Service",PMP,Legal,Executive,Diploma,UI/UX Designer 2 We are hiring a UI/UX Designe...
2,3,Psychologist 3,We are hiring a Psychologist in the Agricultur...,"SQL, AutoCAD, Excel",Six Sigma,Agriculture,Executive,B.Ed,Psychologist 3 We are hiring a Psychologist in...
3,4,Professor 4,We are hiring a Professor in the Finance indus...,"Machine Operation, Leadership, Communication",PMP,Finance,Executive,BCom,Professor 4 We are hiring a Professor in the F...
4,5,Software Developer 5,We are hiring a Software Developer in the Ener...,"Data Analysis, Content Creation, AutoCAD",Google Data Analytics,Energy,Senior,ITI,Software Developer 5 We are hiring a Software ...
...,...,...,...,...,...,...,...,...,...
5995,5996,Insurance Advisor 5996,We are hiring a Insurance Advisor in the E-com...,"Customer Service, Cloud Computing, Problem Sol...",AWS Certified,E-commerce,Senior,B.Tech,Insurance Advisor 5996 We are hiring a Insuran...
5996,5997,DevOps Engineer 5997,We are hiring a DevOps Engineer in the E-comme...,"Cloud Computing, Project Management, Python",Cisco CCNA,E-commerce,Manager,Diploma,DevOps Engineer 5997 We are hiring a DevOps En...
5997,5998,Production Manager 5998,We are hiring a Production Manager in the Defe...,"Sales Strategy, Problem Solving, Communication",IELTS,Defense,Fresher,Diploma,Production Manager 5998 We are hiring a Produc...
5998,5999,Chef 5999,We are hiring a Chef in the Transportation ind...,"Problem Solving, Teaching, Communication",Scrum Master,Transportation,Junior,MBBS,Chef 5999 We are hiring a Chef in the Transpor...
